# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os
from typing import Any, Dict, List, Optional

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import (
    UserMessage,
    SystemMessage,
    ToolMessage,
    AIMessage,
)
from lib.tooling import tool

In [3]:
from dotenv import load_dotenv

# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

import chromadb
from lib.tooling import tool

# Connect to the existing persistent database
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")


@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds the most relevant results in the vector database.

    Args:
        query: A question about the video game industry.

    Returns:
        A list of relevant game records containing Platform, Name,
        YearOfRelease, and Description.
    """
    results = collection.query(
        query_texts=[query],
        n_results=5,
        include=["metadatas", "documents", "distances"],
    )

    games = []

    if not results["metadatas"] or not results["metadatas"][0]:
        return games

    for metadata in results["metadatas"][0]:
        games.append({
            "Platform": metadata.get("Platform", "Unknown"),
            "Name": metadata.get("Name", "Unknown"),
            "YearOfRelease": metadata.get("YearOfRelease", "Unknown"),
            "Description": metadata.get("Description", ""),
        })

    return games

#### Evaluate Retrieval Tool

In [5]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
from pydantic import BaseModel, Field
from lib.llm import LLM
from lib.tooling import tool


class EvaluationReport(BaseModel):
    useful: bool = Field(
        description="Whether the retrieved documents are sufficient to answer the question."
    )
    description: str = Field(
        description="Detailed explanation of the retrieval evaluation and any missing information."
    )


@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> EvaluationReport:
    """
    Based on the user's question and the list of retrieved documents,
    analyze whether the documents are useful for answering the question.

    Args:
        question: Original question from the user.
        retrieved_docs: Documents retrieved from the vector database.

    Returns:
        EvaluationReport containing:
        - useful: Whether the documents are useful to answer the question.
        - description: Detailed explanation of the evaluation result.
    """
    llm = LLM()

    prompt = f"""
    Your task is to evaluate whether the retrieved documents are sufficient
    to answer the user's question.

    User question:
    {question}

    Retrieved documents:
    {retrieved_docs}

    Evaluate the documents based on:
    1. Relevance to the user's question.
    2. Coverage of the information needed to answer it.
    3. Whether the documents provide enough evidence for a reliable answer.
    4. What information is missing, if any.

    Set useful to true only if the documents provide sufficient relevant
    information to answer the question. Otherwise, set it to false.

    Give a detailed explanation so that another agent can decide whether
    to accept the retrieved documents or retrieve more information.

    Return the result in the EvaluationReport format:
    - useful: boolean
    - description: string
    """

    # TODO: Call your LLM and parse its response as EvaluationReport.
    # Replace this line with the structured-output method supported by lib/llm.py.
    response = llm.invoke(prompt, response_format=EvaluationReport)

    return response

#### Game Web Search Tool

In [6]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry.

# TODO: Create game_web_search tool

import os
from tavily import TavilyClient
from lib.tooling import tool


@tool
def game_web_search(question: str) -> str:
    """
    Semantic search: Finds relevant information about the game industry
    on the web.

    Args:
        question: A question about the game industry.

    Returns:
        Search results containing relevant information from the web.
    """
    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

    response = client.search(
        query=question,
        search_depth="advanced",
        topic="general",
        max_results=5,
    )

    results = response.get("results", [])

    if not results:
        return "No relevant web search results were found."

    formatted_results = []

    for result in results:
        formatted_results.append(
            f"Title: {result.get('title', 'N/A')}\n"
            f"URL: {result.get('url', 'N/A')}\n"
            f"Content: {result.get('content', 'N/A')}"
        )

    return "\n\n".join(formatted_results)

### Agent

In [7]:

from lib.agents import Agent

# 1. Define the instructions
game_instructions = """
You are an expert assistant specializing in the video game industry.

Answer questions accurately using reliable evidence.
Use game_web_search to find current information about games.
Use evaluate_retrieval to assess retrieved documents when available.
Never invent facts or sources.
Include source URLs when available.
Keep answers clear, concise, and readable.
"""

# 2. Register the tools
game_tools = [
    evaluate_retrieval,
    game_web_search,
]

# 3. Create the agent
game_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=game_instructions,
    tools=game_tools,
    temperature=0.2,
)

print("Game agent created successfully!")


Game agent created successfully!


In [10]:
from contextlib import redirect_stdout
from io import StringIO
from lib.messages import AIMessage

def ask_game_agent(question: str) -> str:
    # Hide StateMachine debug output
    with redirect_stdout(StringIO()):
        run = game_agent.invoke(question)

    final_state = run.get_final_state()
    messages = final_state.get("messages", [])

    for message in reversed(messages):
        content = getattr(message, "content", None)
        if isinstance(message, AIMessage) and isinstance(content, str) and content.strip():
            return content.strip()

    return "Sorry, no final answer was found. Please check the agent's messages."


questions = [
    "When were Pokémon Gold and Silver released?",
    "Which was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for i, question in enumerate(questions, start=1):
    print(f"\n{'=' * 60}")
    print(f"Question {i}: {question}\n")
    print(ask_game_agent(question))

print("\n" + "=" * 60)



Question 1: When were Pokémon Gold and Silver released?

Pokémon Gold and Silver were released on the following dates:

- **Japan**: November 21, 1999
- **North America**: October 15, 2000
- **Europe**: April 6, 2001

For more details, you can visit the [Wikipedia page](https://en.wikipedia.org/wiki/Pok%C3%A9mon_Gold_and_Silver).

Question 2: Which was the first 3D platformer Mario game?

The first 3D platformer Mario game is "Super Mario 64," which was released for the Nintendo 64 on June 23, 1996, in Japan and later in North America on September 29, 1996. It was groundbreaking for its time, introducing players to a fully 3D environment and innovative gameplay mechanics.

Question 3: Was Mortal Kombat X released for PlayStation 5?

Mortal Kombat X was not released natively for the PlayStation 5. However, it is available to play on the PS5 through the PlayStation Plus Collection, which includes the game as part of its offerings for PS4 titles. The game was originally released on April

### (Optional) Advanced

In [13]:
# TODO: Update your agent with long-term memory
import sqlite3
from lib.messages import UserMessage, AIMessage

memory_db = "udaplay_memory.db"


def init_memory():
    with sqlite3.connect(memory_db) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS conversation_memory (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT NOT NULL,
                role TEXT NOT NULL,
                content TEXT NOT NULL
            )
        """)


def save_memory(session_id, role, content):
    with sqlite3.connect(memory_db) as conn:
        conn.execute(
            """INSERT INTO conversation_memory
               (session_id, role, content)
               VALUES (?, ?, ?)""",
            (session_id, role, content),
        )


def load_memory(session_id, limit=10):
    with sqlite3.connect(memory_db) as conn:
        rows = conn.execute(
            """SELECT role, content FROM conversation_memory
               WHERE session_id = ?
               ORDER BY id DESC LIMIT ?""",
            (session_id, limit),
        ).fetchall()

    return list(reversed(rows))


init_memory()
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

from typing import TypedDict, Any
from langgraph.graph import StateGraph, START, END


# State shared between all nodes
class GameAgentState(TypedDict, total=False):
    query: str
    retrieved_games: list
    evaluation: dict
    web_results: list
    answer: str


def build_agent(
    retrieve_game,
    evaluate_retrieval,
    game_web_search,
    generate_answer,
):
    # Node 1: Retrieve games from local ChromaDB
    def retrieve_node(state: GameAgentState):
        results = retrieve_game(state["query"])
        return {"retrieved_games": results}

    # Node 2: Evaluate retrieval quality
    def evaluate_node(state: GameAgentState):
        evaluation = evaluate_retrieval({
            "query": state["query"],
            "results": state.get("retrieved_games", []),
        })
        return {"evaluation": evaluation}

    # Node 3: Search the web only if local evidence is insufficient
    def web_search_node(state: GameAgentState):
        results = game_web_search(state["query"])
        return {"web_results": results}

    # Node 4: Generate the final answer
    def answer_node(state: GameAgentState):
        answer = generate_answer(
            query=state["query"],
            games=state.get("retrieved_games", []),
            web_results=state.get("web_results", []),
        )
        return {"answer": answer}

    # Conditional routing after evaluation
    def route_after_evaluation(state: GameAgentState):
        evaluation = state.get("evaluation", {})
        if evaluation.get("sufficient", False):
            return "answer"
        return "web_search"

    # Define the graph
    graph = StateGraph(GameAgentState)

    graph.add_node("retrieve_game", retrieve_node)
    graph.add_node("evaluate_retrieval", evaluate_node)
    graph.add_node("game_web_search", web_search_node)
    graph.add_node("generate_answer", answer_node)

    graph.add_edge(START, "retrieve_game")
    graph.add_edge("retrieve_game", "evaluate_retrieval")

    graph.add_conditional_edges(
        "evaluate_retrieval",
        route_after_evaluation,
        {
            "answer": "generate_answer",
            "web_search": "game_web_search",
        },
    )

    graph.add_edge("game_web_search", "generate_answer")
    graph.add_edge("generate_answer", END)

    return graph.compile()
